# 使用矩阵分解生成电影推荐器

> https://learn.microsoft.com/zh-cn/dotnet/machine-learning/tutorials/movie-recommendation
>
> https://www.kaggle.com/code/user215638/movie-recommendation2


In [2]:
!pip install surprise

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import SVD, SVDpp, Reader, Dataset, accuracy
from surprise.model_selection import GridSearchCV
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# === 1. 加载数据
train_df = pd.read_csv('/kaggle/input/movie-data/recommendation-ratings-train.txt', sep=',', header=0)
test_df = pd.read_csv('/kaggle/input/movie-data/recommendation-ratings-test.txt', sep=',', header=0)

# === 2. 可视化分析

# 评分分布
plt.figure(figsize=(6, 4))
sns.histplot(train_df['rating'], bins=10, kde=True)
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

# 用户活跃度分布
user_rating_counts = train_df.groupby('userId')['rating'].count()
plt.figure(figsize=(6, 4))
sns.histplot(user_rating_counts, bins=30)
plt.title('Number of Ratings per User')
plt.xlabel('Ratings per User')
plt.ylabel('Number of Users')
plt.show()

# 电影热度分布
item_rating_counts = train_df.groupby('movieId')['rating'].count()
plt.figure(figsize=(6, 4))
sns.histplot(item_rating_counts, bins=30)
plt.title('Number of Ratings per Movie')
plt.xlabel('Ratings per Movie')
plt.ylabel('Number of Movies')
plt.show()

In [ ]:
# === 3. 数据清洗（去除冷启动用户/物品）
min_ratings = 5
user_counts = train_df['userId'].value_counts()
item_counts = train_df['movieId'].value_counts()
valid_users = user_counts[user_counts >= min_ratings].index
valid_items = item_counts[item_counts >= min_ratings].index

filtered_train_df = train_df[
    train_df['userId'].isin(valid_users) & train_df['movieId'].isin(valid_items)
].copy()

filtered_test_df = test_df[
    test_df['userId'].isin(valid_users) & test_df['movieId'].isin(valid_items)
].copy()

# === 4. 构建 Surprise 数据结构
reader = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(filtered_train_df[['userId', 'movieId', 'rating']], reader)
trainset = train_data.build_full_trainset()
testset = list(filtered_test_df[['userId', 'movieId', 'rating']].itertuples(index=False, name=None))

# === 5. 定义参数搜索范围
param_grid = {
    'n_epochs': [20, 40],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.02, 0.1],
    'n_factors': [50, 100]
}

# === 6. SVD 模型训练与评估
print("🔍 正在搜索 SVD 最优参数...")
gs_svd = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs_svd.fit(train_data)
best_svd = gs_svd.best_estimator['rmse']
best_svd.fit(trainset)
pred_svd = best_svd.test(testset)
rmse_svd = accuracy.rmse(pred_svd, verbose=True)

# === 7. SVD++ 模型训练与评估
print("🔍 正在搜索 SVD++ 最优参数...")
gs_svdpp = GridSearchCV(SVDpp, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs_svdpp.fit(train_data)
best_svdpp = gs_svdpp.best_estimator['rmse']
best_svdpp.fit(trainset)
pred_svdpp = best_svdpp.test(testset)
rmse_svdpp = accuracy.rmse(pred_svdpp, verbose=True)

# === 8. 模型性能对比
print("\n📊 模型对比结果:")
print(f"SVD    : RMSE = {rmse_svd:.4f}")
print(f"SVD++  : RMSE = {rmse_svdpp:.4f}")

## ✅ 如何理解 RMSE 的结果？

| RMSE 范围    | 解读              |
| ---------- | --------------- |
| < 1.0      | 非常好，预测评分非常接近真实值 |
| 1.0 \~ 1.5 | 可接受，效果较好        |
| 1.5 \~ 2.5 | 中等，可能存在欠拟合或数据稀疏 |
| > 2.5      | 效果较差，模型可能欠缺泛化能力 |